Part A — Set Up Google Colab

In [ ]:
!pip install -U transformers accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling pef

Part B — Import Libraries

In [ ]:
import torch
import time
import logging
import warnings
import os

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from datasets import load_dataset, DatasetDict

from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType
)

warnings.filterwarnings("ignore")

# Suppress Hugging Face and bitsandbytes warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)


Part C — Define Model and Tokenizer

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Part D — Reusable Inference Function

In [ ]:
def generate_response(model, prompt, max_new_tokens=100):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )

    inference_time = time.time() - start_time

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response, inference_time

Part E — Load Baseline Model

In [ ]:
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_fp16.eval()

if torch.cuda.is_available():
  print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
  print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Allocated: 0.9202375411987305 GB
Reserved: 0.931640625 GB


Part F — Baseline Inference

In [ ]:
prompts = [
    "Generate a simple recipe for Chocolate Cake. Include Ingredients and Directions.",
    "Generate a recipe for Vegetable Fried Rice. Include Ingredients and Directions.",
    "Generate a recipe for Chicken Pasta. Include Ingredients and Directions."
]

In [ ]:
baseline_results = []

for prompt in prompts:
    response, t = generate_response(
        model_fp16,
        prompt,
        max_new_tokens=100
    )

    baseline_results.append({
        "prompt": prompt,
        "response": response,
        "time": t
    })

    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("TIME:", t, "seconds")
    print("-" * 80)

PROMPT: Generate a simple recipe for Chocolate Cake. Include Ingredients and Directions.
RESPONSE:  

Ingredients:
- 2 cups all-purpose flour
- 1 cup granulated sugar
- .5 cup unsalted butter, softened
-. 3/4 cup white vinegar
-1 tsp baking soda
-2 tsp salt
-3/8 tsp ground cinnamon
- Pinch of nutmeg
-Pinch of vanilla extract

Directions:

1. Preheat oven to 375 degrees F (190 degrees C). Grease and flour two 9
TIME: 5.166383743286133 seconds
--------------------------------------------------------------------------------
PROMPT: Generate a recipe for Vegetable Fried Rice. Include Ingredients and Directions.
RESPONSE:  

Ingredients:
- 1 cup cooked brown rice
- 2 cups mixed vegetables (such as carrots, peas, corn, bell peppers)
- 3 tablespoons vegetable oil
- Salt and pepper to taste

Directions:

1. In a large skillet or wok, heat the vegetable oil over medium-high heat.
2. Add the vegetables and sauté until they are tender-crisp, about 5 minutes.
3. Add in the cooked browned rice and 

In [ ]:
baseline_avg_time = sum(
    x["time"] for x in baseline_results
) / len(baseline_results)

print("Baseline FP16 Average Inference Time:",
      baseline_avg_time,
      "seconds")

Baseline FP16 Average Inference Time: 3.9295756816864014 seconds


Part G — 8-bit Quantization

In [ ]:
quant_config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config_8bit,
    device_map="auto"
)

model_int8.eval()

if torch.cuda.is_available():
  print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
  print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Allocated: 1.5172672271728516 GB
Reserved: 1.55859375 GB


In [ ]:
int8_results = []

for prompt in prompts:
    response, t = generate_response(
        model_int8,
        prompt,
        max_new_tokens=100
    )

    int8_results.append({
        "prompt": prompt,
        "response": response,
        "time": t
    })

    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("TIME:", t, "seconds")
    print("-" * 80)

PROMPT: Generate a simple recipe for Chocolate Cake. Include Ingredients and Directions.
RESPONSE:  

Ingredients:
- 2 cups all-purpose flour
- 1 cup granulated sugar
- .5 cups unsalted butter, softened
-. 3/4 cup brown sugar
. 1 teaspoon baking soda
.1 teaspoon salt
-1/2 cup milk
-2 large eggs
-3/8 cup vegetable oil
- pinch of vanilla extract

Directions:

1. Preheat oven to 350 degrees F (175 degrees C). Grease
TIME: 14.664647102355957 seconds
--------------------------------------------------------------------------------
PROMPT: Generate a recipe for Vegetable Fried Rice. Include Ingredients and Directions.
RESPONSE:  

Vegetable Fried Rice Recipe

Ingredients:
- 1 cup cooked rice
- 2 cups mixed vegetables (such as bell peppers, carrots, onions, mushrooms)
- 3 tablespoons vegetable oil
- Salt to taste
- Optional: 1 tablespoon soy sauce or tamari for flavor

Directions:

1. In a large skillet, heat the vegetable oil over medium-high heat.
2. Add the vegetables and sauté until they a

In [ ]:
int8_avg_time = sum(
    x["time"] for x in int8_results
) / len(int8_results)

print("INT8 Average Inference Time:",
      int8_avg_time,
      "seconds")

INT8 Average Inference Time: 16.341591278711956 seconds


Part H — 4-bit Quantization

In [ ]:
quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config_4bit,
    device_map="auto"
)

model_int4.eval()

if torch.cuda.is_available():
  print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
  print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Allocated: 1.94352388381958 GB
Reserved: 1.958984375 GB


In [ ]:
int4_results = []

for prompt in prompts:
    response, t = generate_response(
        model_int4,
        prompt,
        max_new_tokens=100
    )

    int4_results.append({
        "prompt": prompt,
        "response": response,
        "time": t
    })

    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("TIME:", t, "seconds")
    print("-" * 80)

PROMPT: Generate a simple recipe for Chocolate Cake. Include Ingredients and Directions.
RESPONSE:  Here's a simple chocolate cake recipe:

Ingredients:
- 2 cups all-purpose flour
- 1 cup unsalted butter, softened
- Large egg
- Pinch of salt
- One cup sugar
- Two eggs
- Three cups of granulated sugar
 - One cup of cocoa powder
- Four cups of milk (unsweetened)
- One teaspoon vanilla extract

Directions:
1. Preheat your oven to 350°F (175°C).
2.
TIME: 5.96501088142395 seconds
--------------------------------------------------------------------------------
PROMPT: Generate a recipe for Vegetable Fried Rice. Include Ingredients and Directions.
RESPONSE:  Here's a simple recipe for a delicious Vegetable Fried rice:

Ingredients:
- 1 cup of cooked brown rice
- 2 tablespoons of vegetable oil
- Salt, to taste
- Fresh vegetables such as carrots, peas, and bell peppers

Directions:

1. In a large skillet or wok, heat the oil over medium-high heat.

2. Add the vegetables (peas, carrots, and/or b

In [ ]:
int4_avg_time = sum(
    x["time"] for x in int4_results
) / len(int4_results)

print("INT4 Average Inference Time:",
      int4_avg_time,
      "seconds")

INT4 Average Inference Time: 5.782664696375529 seconds


Part I — Compare Quantized Models

| Configuration | Precision | GPU Memory (Allocated) | Avg. Time | Output Quality | Remarks |
|---|---|---|---|---|---|
| Baseline | FP16 | 0.92 GB | 3.93 s | Coherent, fluent, correct recipe format | Fastest of the three; full-precision reference |
| Quantized 8-bit | INT8 | 1.52 GB* | 16.34 s | Coherent, near-identical structure to baseline | Slowest despite being higher precision than INT4 |
| Quantized 4-bit / NF4 | INT4/NF4 | 1.94 GB* | 5.78 s | Coherent, slightly more verbose/repetitive phrasing | Much faster than INT8, close to baseline speed |


Part J — Prepare the Model for QLoRA

In [ ]:
from peft import prepare_model_for_kbit_training

model_qlora = prepare_model_for_kbit_training(model_int4)

Part K — Configure and Attach LoRA


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_qlora = get_peft_model(
    model_qlora,
    lora_config
)

model_qlora.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Part L — Load a Small Task-Specific Dataset


In [ ]:
from datasets import load_dataset, DatasetDict

print("Loading Recipe Dataset...")

ds_recipes = load_dataset(
    "SandhyaKilari/RecipeNLG_dataset"
)

selected_recipes = ds_recipes["train"].select(range(500))


def format_recipe(example):
    title = example.get("title", "Unknown Recipe")
    ingredients = example.get("ingredients", "Unknown Ingredients")
    directions = example.get("directions", "Unknown Directions")

    formatted_text = (
        f"### Instruction:\n"
        f"Generate a recipe for {title}.\n\n"
        f"### Response:\n"
        f"Recipe: {title}\n"
        f"Ingredients: {ingredients}\n"
        f"Directions: {directions}"
    )

    return {"text": formatted_text}


print("Formatting recipes...")

train_recipes = selected_recipes.map(
    format_recipe,
    remove_columns=selected_recipes.column_names
)

ds = DatasetDict({
    "train": train_recipes.shuffle(seed=42)
})

print("Dataset loaded, formatted and shuffled.")
print(ds)
print(ds["train"][0]["text"])

Loading Recipe Dataset...


README.md:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

RecipeNLG_dataset.csv: reconstructing file:   0%|          |  0.00B / 2.29GB            

RecipeNLG_dataset.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2231142 [00:00<?, ? examples/s]

Formatting recipes...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset loaded, formatted and shuffled.
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 500
    })
})
### Instruction:
Generate a recipe for Yum Yum Salad.

### Response:
Recipe: Yum Yum Salad
Ingredients: ["1 can cherry pie filling", "1 large can drained, crushed pineapple", "1 can Borden milk", "3/4 c. chopped pecans", "1 (8 oz.) Cool Whip"]
Directions: ["Mix together and let stand overnight in refrigerator."]


Part M — Tokenize the Dataset


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = ds.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 500
    })
})


Part N — QLoRA Training Configuration


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qlora_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=25,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model_qlora,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

In [ ]:
print("Starting QLoRA training...")

train_result = trainer.train()

print("QLoRA training completed.")

Starting QLoRA training...
{'loss': '1.427', 'grad_norm': '2.841', 'learning_rate': '0.0001904', 'epoch': '0.1'}
{'loss': '1.381', 'grad_norm': '3.068', 'learning_rate': '0.0001804', 'epoch': '0.2'}
{'loss': '1.355', 'grad_norm': '3.059', 'learning_rate': '0.0001704', 'epoch': '0.3'}
{'loss': '1.385', 'grad_norm': '3.188', 'learning_rate': '0.0001604', 'epoch': '0.4'}
{'loss': '1.383', 'grad_norm': '3.021', 'learning_rate': '0.0001504', 'epoch': '0.5'}
{'loss': '1.319', 'grad_norm': '2.8', 'learning_rate': '0.0001404', 'epoch': '0.6'}
{'loss': '1.268', 'grad_norm': '2.698', 'learning_rate': '0.0001304', 'epoch': '0.7'}
{'loss': '1.379', 'grad_norm': '3.108', 'learning_rate': '0.0001204', 'epoch': '0.8'}
{'loss': '1.369', 'grad_norm': '2.693', 'learning_rate': '0.0001104', 'epoch': '0.9'}
{'loss': '1.3', 'grad_norm': '2.458', 'learning_rate': '0.0001004', 'epoch': '1'}
{'loss': '1.304', 'grad_norm': '2.887', 'learning_rate': '9.04e-05', 'epoch': '1.1'}
{'loss': '1.293', 'grad_norm': '2.

In [ ]:
print("Training statistics:")
print(train_result.metrics)

Training statistics:
{'train_runtime': 174.5125, 'train_samples_per_second': 5.73, 'train_steps_per_second': 2.865, 'train_loss': 1.3275626602172852, 'epoch': 2.0}


Part O — Save and Evaluate the Adapter

In [ ]:
model_qlora.save_pretrained("./qlora_adapter")
tokenizer.save_pretrained("./qlora_adapter")

('./qlora_adapter/tokenizer_config.json',
 './qlora_adapter/chat_template.jinja',
 './qlora_adapter/tokenizer.json')

In [ ]:
model_qlora.eval()
model_qlora.config.use_cache = True

In [ ]:
qlora_results = []

qlora_prompts = [
    "### Instruction:\nGenerate a recipe for Chocolate Cake.\n\n### Response:\n",
    "### Instruction:\nGenerate a recipe for Vegetable Fried Rice.\n\n### Response:\n",
    "### Instruction:\nGenerate a recipe for Chicken Pasta.\n\n### Response:\n"
]

for prompt in qlora_prompts:
    response, t = generate_response(
        model_qlora,
        prompt,
        max_new_tokens=100
    )

    qlora_results.append({
        "prompt": prompt,
        "response": response,
        "time": t
    })

    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("TIME:", t, "seconds")
    print("-" * 80)

PROMPT: ### Instruction:
Generate a recipe for Chocolate Cake.

### Response:

RESPONSE: Recipe: Chocolate Cake
Ingredients: ["1 c. sugar", "2 eggs", "3/4 c. butter or margarine", "1 1/2 c. flour", "pinch of salt", "8 oz. chocolate chips"]
Directions: ["Cream together the sugar and butter.", "Add the egg mixture, one at a time.", "Beat until light and fluffy.", "Gradually add the flour and salt.", "Mix well.", "Pour into a greased and fl
TIME: 12.744637250900269 seconds
--------------------------------------------------------------------------------
PROMPT: ### Instruction:
Generate a recipe for Vegetable Fried Rice.

### Response:

RESPONSE: Recipe: Vegetable Fried rice
Ingredients: ["1 (8 oz.) can cream of mushroom soup", "2 Tbsp. butter or margarine", "1 c. chopped onion", "3/4 c. grated carrots, 1/4 tsp. salt and pepper", "dash of garlic powder", "6 to 7 green peas, cut in thin pieces", "5 to 6 green bell peppers, cut into small pieces", "'sliced" carrot, 2 cloves garlic,
TIME: 6.8

In [ ]:
qlora_avg_time = sum(
    x["time"] for x in qlora_results
) / len(qlora_results)

print("QLoRA Average Inference Time:",
      qlora_avg_time,
      "seconds")

QLoRA Average Inference Time: 7.798505147298177 seconds


Observation Table

| Run | Precision | LoRA | Trainable Params | Mem Allocated | Mem Reserved | Approx Model Mem (calculation) | Avg. Time | Task Output |
|---|---|---|---|---|---|---|---|---|
| Baseline | FP16 | No | 494,573,440 (100%) | 0.9202 GB | 0.9316 GB | 0.9202 − 0 = **0.9202 GB** | 3.93 s | Fluent recipe generations for all 3 prompts |
| INT8 | 8-bit | No | 494,573,440 (100%) | 1.5173 GB | 1.5586 GB | 1.5173 − 0.9202 = **0.5971 GB** | 16.34 s | Fluent, near-identical output to baseline |
| INT4 | 4-bit/NF4 | No | 494,573,440 (100%) | 1.9435 GB | 1.9590 GB | 1.9435 − 1.5173 = **0.4262 GB** | 5.78 s | Fluent, minor phrasing differences, still coherent |
| QLoRA | 4-bit/NF4 | Yes | 540,672 (0.109%) | - | - | 0.4262 GB (base) + ~1 MB (adapter)| 7.79 s | Recipes now follow fine-tuned `### Instruction/### Response` format learned from RecipeNLG |

**Note:** Allocated/Reserved are cumulative since each model was loaded without freeing the previous one — "Approx Model Mem" = current stage's Allocated − previous stage's Allocated, isolating what each newly loaded model actually added to the GPU.

## Results and Discussion

### 1. Effect of reduced precision on memory and inference behavior

Reduced precision decreases the number of bits used to store the model weights, so the model requires less GPU memory. In our experiment, the quantized models used less additional memory compared to the FP16 model.\
The inference time did not always decrease directly with lower precision. The FP16 model took around **3.93 seconds**, INT8 took around **16.34 seconds**, and INT4 took around **5.78 seconds** on average. This shows that lower precision mainly helped in reducing memory usage, while inference speed also depends on the hardware and quantization implementation.

### 2. Why can QLoRA fine-tune a quantized model without updating all base-model parameters?

QLoRA keeps the main model in a **4-bit quantized form and freezes its parameters**. Instead of updating the complete base model, small **LoRA adapter parameters** are added to selected layers and only these parameters are trained.\
Therefore, the model can learn task-specific information without changing all the original parameters. This reduces the amount of memory and computation required for fine-tuning. In our experiment, only a very small portion of the total parameters were trainable.\
In simple terms, the quantized base model provides the original knowledge, while the LoRA adapters learn the new task on top of it.

### 3. Which configuration used the least GPU memory?

The **4-bit/NF4 configuration** used the least additional GPU memory. Its approximate additional memory usage was **0.4262 GB**, while the INT8 configuration used around **0.5971 GB**.\
The memory shown directly in the observation table was cumulative, so the difference between consecutive runs was used to estimate the additional memory required by each configuration.

### 4. How did 8-bit and 4-bit quantization differ in resource usage?

The 4-bit model required less memory than the 8-bit model because each parameter was represented using fewer bits.\
In our experiment, INT8 used approximately **0.5971 GB** of additional memory, while INT4/NF4 used approximately **0.4262 GB**. The INT4 model also had a lower average inference time than INT8 in our run.\
Therefore, 4-bit quantization gave better memory efficiency in this experiment.

### 5. Did the generated outputs remain similar after quantization?

Yes, the outputs remained mostly similar after quantization. The INT8 model generated responses that were quite close to the FP16 model.\
The INT4/NF4 model also produced relevant and understandable outputs, although there were some small differences in wording and formatting. Overall, the main information in the generated responses was still preserved.\
This shows that quantization reduced the memory requirement without causing a major change in the generated output for the prompts used in the experiment.

### 6. What is the role of NF4?

NF4 stands for **NormalFloat 4-bit**. It is a 4-bit data type commonly used in QLoRA.\
It is designed to store neural network weights efficiently while trying to preserve the important information in the original weights. In our experiment, NF4 was used for the 4-bit model and helped reduce memory usage.

### 7. Why is the base model kept frozen in QLoRA?

The base model is kept frozen so that we do not need to update all of its parameters during training. The original model already contains a large amount of learned information, so changing all of it would require much more GPU memory and computation.\
Instead, QLoRA trains only the small LoRA adapter parameters. This makes fine-tuning possible even when GPU resources are limited.

### 8. How is QLoRA different from ordinary LoRA?

The main difference is the way the base model is stored.
- **LoRA:** Standard-precision base model + LoRA adapters
- **QLoRA:** 4-bit quantized base model + LoRA adapters

Both methods keep the base model frozen and train only the small LoRA adapters. QLoRA additionally quantizes the base model, which further reduces memory usage.

### 9. Why should the same prompts be used for a fair comparison?

The same prompts should be used so that all the models are tested under the same conditions.\
If different prompts are given to FP16, INT8, and INT4 models, then it becomes difficult to know whether the difference in output or inference time is because of quantization or because the prompts were different.\
In our experiment, the same recipe-related prompts were used for the different model configurations. This makes the comparison more consistent and helps us understand the actual effect of quantization.



### Overall Observation

Overall, the experiment showed that quantization can significantly reduce the memory required by the model while keeping the generated outputs reasonably similar. The **4-bit/NF4 configuration used the least additional memory**, and QLoRA allowed the quantized model to be fine-tuned by training only a small number of LoRA parameters instead of updating the complete base model.